In [ ]:
import numpy as np
from causallearn.search.ConstraintBased.PC import pc, fci
from causallearn.utils.cit import fisherz, chisq, gsq, kci
from causallearn.graph.SHD import SHD
from causallearn.utils.DAG2CPDAG import dag2cpdag
from causallearn.utils.DAG2PAG import dag2pag
from causallearn.utils.TXT2GeneralGraph import txt2generalgraph

### Load the datasets

In [ ]:
linear_data_path = "./TestData/data_linear_10.txt"
discrete_data_path = "./TestData/data_discrete_10.txt"
graph_path = "./TestData/graph.10.txt" 

linear_data = np.loadtxt(linear_data_path, skiprows=1)
discrete_data = np.loadtxt(discrete_data_path, skiprows=1)
DAG = txt2generalgraph(graph_path)
CPDAG = dag2cpdag(DAG)
PAG = dag2pag(DAG)

#### Manually try PC with different indep_test

In [ ]:
# Linear data, only Fisherz works
model = pc(linear_data, alpha=0.05, indep_test="fisherz")
shd = SHD(CPDAG, model.G)
print(shd.get_shd())

In [ ]:
# Discrete data
# model = pc(discrete_data, alpha=0.05, indep_test="fisherz")
# shd = SHD(CPDAG, model.G)
# print(shd.get_shd())

model = pc(discrete_data, alpha=0.05, indep_test="chisq")
shd = SHD(CPDAG, model.G)
print(shd.get_shd())

model = pc(discrete_data, alpha=0.05, indep_test="gsq")
shd = SHD(CPDAG, model.G)
print(shd.get_shd())

#### Manually try PC with different uc_rule

In [ ]:
# uc_sepset
model = pc(linear_data, alpha=0.05, indep_test="fisherz", uc_rule=0)
shd = SHD(CPDAG, model.G)
print(shd.get_shd())

# maxP
model = pc(linear_data, alpha=0.05, indep_test="fisherz", uc_rule=1)
shd = SHD(CPDAG, model.G)
print(shd.get_shd())

# definiteMaxP
model = pc(linear_data, alpha=0.05, indep_test="fisherz", uc_rule=2)
shd = SHD(CPDAG, model.G)
print(shd.get_shd())

#### HPO: PC with alpha

In [ ]:
from ConfigSpace import ConfigurationSpace
from ConfigSpace.hyperparameters import UniformIntegerHyperparameter, CategoricalHyperparameter, UniformFloatHyperparameter
from smac.facade.smac_bb_facade import SMAC4BB
from smac.facade.smac_hpo_facade import SMAC4HPO
from smac.facade.smac_ac_facade import SMAC4AC
from smac.scenario.scenario import Scenario

In [ ]:
def train_pc_algorithm(config):
    """ 
    Objective function (minimizes)
    Input: configuration
    Output: SHD value
    
    """
    model = pc(linear_data, alpha=config["alpha"], indep_test='fisherz')
    shd = SHD(CPDAG, model.G)
    return shd.get_shd()

In [ ]:
# Define configuration space
configspace = ConfigurationSpace()
alpha = UniformFloatHyperparameter("alpha", 0, 0.5, default_value=0.05)
configspace.add_hyperparameter(alpha)
#print("Sample two configurations: \n", configspace.sample_configuration(2))

scenario = Scenario({
    "run_obj": "quality",   # optimize objective function (either quality or runtime)
    "runcount-limit": 10,    # max number of function evaluations
    "cs": configspace, 
#     "deterministic": False,
})

def_value = train_pc_algorithm(configspace.get_default_configuration())
print("Default Value: %.2f" % def_value)

smac = SMAC4HPO(
    scenario=scenario, 
    rng=np.random.RandomState(42),
    tae_runner=train_pc_algorithm,
)

best_found_config = smac.optimize()
best_value = train_pc_algorithm(best_found_config)
print("Best found configuration: \n" % best_found_config)
print("Optimized Value: %.2f" % best_value)

# Show runhistory
# rh = smac.get_runhistory()
# for (config_id, instance_id, seed, budget), (cost, time, status, starttime, endtime, additional_info) in rh.data.items():
#     config = rh.ids_config[config_id]
#     print(config)

#### Manually try GES with different score_func

In [ ]:
from causallearn.search.ScoreBased.GES import ges

In [ ]:
model = ges(linear_data, score_func="local_score_BIC")
shd = SHD(CPDAG, model['G'])
shd.get_shd()